In [0]:
!pip install --upgrade "mlflow[databricks]==3.8.1"
!pip install "backoff==2.2.1"
!pip install "databricks-openai==0.8.0"

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../Includes/_common

In [0]:
%run ./Classroom-Setup-Common

In [0]:
DA = DBAcademyHelper()
DA.init()

In [0]:
import json
from pathlib import Path


class DemoSetup:
    def __init__(
        self,
        catalog_name: str = None,
        schema_name: str = "demo_agent_eval",
        agent_name: str = "airbnb_eval_agent"
    ):
        self.agent_name = agent_name

    def run(self) -> None:
        print("=" * 60)
        print("Starting Databricks Agent Demo Setup")
        print("=" * 60)

        self.dev_lab_setup()

    def dev_lab_setup(self):
        spark.sql(f"USE CATALOG {DA.catalog_name}")
        print(f"Using catalog: {DA.catalog_name}")
        spark.sql(f"USE SCHEMA {DA.schema_name}")
        print(f"Using schema: {DA.schema_name}")
        return None 
    
    def get_env_vars(self):
        print(f"Using catalog: {DA.catalog_name}")
        print(f"Using schema: {DA.schema_name}")
        print(f"Getting agent name: {self.agent_name}")
        return DA.catalog_name, DA.schema_name, self.agent_name

In [0]:
demo_setup = DemoSetup(
    # catalog_name = <FILL_IN> # Pass only if you created a new catalog in 01 Demo - Agent Setup
)

# Run the build-pipeline
demo_setup.run()

# Get the catalog, schema, and agent string values
catalog_name, schema_name, agent_name = demo_setup.get_env_vars()

In [0]:
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

# Set the location of your experiment
EXPERIMENT_NAME = f"/Users/{username}/airbnb_experiment"

# Print the results
print(f"Your username is: {username}")
print(f"The location of your experiment: {EXPERIMENT_NAME}")

In [0]:
import mlflow

# Enable MLflow's autologging to instrument your application with Tracing
#mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_NAME)

In [0]:
# Set the registry to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Load the model using the UC path and alias
alias = "champion"

UC_MODEL_NAME = f"{catalog_name}.{schema_name}.{agent_name}"

print(UC_MODEL_NAME)

# Load by alias (recommended for production)
agent = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_NAME}@{alias}")

In [0]:
development_config = "../artifacts/configs/agent_eval_config.yaml"

config = mlflow.models.ModelConfig(development_config=development_config)

correctness_eval_endpoint = config.get('CORRECTNESS_EVAL_ENDPOINT')
safety_eval_endpoint = config.get('SAFETY_EVAL_ENDPOINT')

print(f"Correctness Endpoint: {correctness_eval_endpoint}")
print(f"Safety Endpoint: {safety_eval_endpoint}")